# Noah Miller 

**Cluster 5 Stacking Model**

**Group 2** | Cluster 5: 998 companies, 50 bankrupt (5.01% rate)

Note: moderate class imbalance with enough positive samples for reliable CV estimates.

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, AdaBoostClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CLUSTER_DIR = os.path.join(DATA_DIR, 'clusters')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
SUBGROUP_MODEL_DIR = os.path.join(MODEL_DIR, 'subgroup_models')
os.makedirs(SUBGROUP_MODEL_DIR, exist_ok=True)

In [16]:
def custom_accuracy(y_true, y_pred):
    """Project metric: acc = TT / (TT + TF) = recall of bankrupt class.
    TT = correctly predicted bankrupt, TF = bankrupt predicted as non-bankrupt."""
    tt = ((y_true == 1) & (y_pred == 1)).sum()
    tf = ((y_true == 1) & (y_pred == 0)).sum()
    if tt + tf == 0:
        return 0.0
    return tt / (tt + tf)

def sparsity_check(y_pred, threshold=0.20):
    """Check if < 20% of predictions are bankrupt."""
    rate = y_pred.mean()
    return rate < threshold, rate

def evaluate_model(y_true, y_pred, label=''):
    """Print evaluation metrics."""
    acc = custom_accuracy(y_true, y_pred)
    ok, rate = sparsity_check(y_pred)
    cm = confusion_matrix(y_true, y_pred)
    print(f'--- {label} ---')
    print(f'Custom Accuracy (TT/(TT+TF)): {acc:.4f}')
    print(f'Sparsity: {rate:.4f} ({"PASS" if ok else "FAIL"})')
    print(f'Confusion Matrix:\n{cm}')
    print(f'Classification Report:\n{classification_report(y_true, y_pred, zero_division=0)}')
    return acc, rate

---
## Load & Inspect Data

In [17]:
# Load cluster 5 data
df5 = pd.read_csv(os.path.join(CLUSTER_DIR, 'cluster_5.csv'))
y5 = df5['Bankrupt?'].values
X5 = df5.drop(columns=['Bankrupt?']).values
feature_names_5 = df5.drop(columns=['Bankrupt?']).columns.tolist()

print(f'Cluster 5: {X5.shape[0]} samples, {X5.shape[1]} features')
print(f'Bankrupt: {y5.sum()} ({y5.mean():.4f})')

Cluster 5: 998 samples, 95 features
Bankrupt: 50 (0.0501)


## Feature Selection

In [24]:
# Feature selection
mi_scores_5 = mutual_info_classif(X5, y5, random_state=RANDOM_STATE)
mi_series_5 = pd.Series(mi_scores_5, index=feature_names_5).sort_values(ascending=False)

# Sweep showed N=10 maximizes rank score (0.660)
N_FEATURES_5 = 10
top_features_5 = mi_series_5.head(N_FEATURES_5).index.tolist()
X5_sel = df5[top_features_5].values

print(f'Selected {N_FEATURES_5} features for Cluster 5:')
for i, (feat, score) in enumerate(mi_series_5.head(N_FEATURES_5).items()):
    print(f'  {i+1:2d}. {feat[:55]:55s}  MI = {score:.4f}')

Selected 10 features for Cluster 5:
   1. Borrowing dependency                                     MI = 0.0289
   2. Current Liabilities/Equity                               MI = 0.0269
   3. Current Liability to Equity                              MI = 0.0269
   4. Cash flow rate                                           MI = 0.0265
   5. Current Liability to Assets                              MI = 0.0258
   6. Non-industry income and expenditure/revenue              MI = 0.0184
   7. Working capitcal Turnover Rate                           MI = 0.0177
   8. Persistent EPS in the Last Four Seasons                  MI = 0.0177
   9. Operating Funds to Liability                             MI = 0.0172
  10. Net worth/Assets                                         MI = 0.0165


## Cross-Validation

In [25]:
# Cross-validate with SMOTE + probability threshold tuning
# 5-fold stratified CV (each fold gets ~10 positives)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y5_cv_probas = np.zeros(len(y5))

for fold, (train_idx, val_idx) in enumerate(cv5.split(X5_sel, y5)):
    X_train_cv, X_val_cv = X5_sel[train_idx], X5_sel[val_idx]
    y_train_cv, y_val_cv = y5[train_idx], y5[val_idx]
    
    n_pos = int(y_train_cv.sum())
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, n_pos - 1),
                  sampling_strategy=min(0.3, 1.0))
    X_train_res, y_train_res = smote.fit_resample(X_train_cv, y_train_cv)
    
    stacker_fold = StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
            ('gb', GradientBoostingClassifier(n_estimators=80, max_depth=3, learning_rate=0.05, min_samples_leaf=10, random_state=RANDOM_STATE)),
            ('et', ExtraTreesClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ],
        final_estimator=LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=RANDOM_STATE),
        cv=3,
        stack_method='predict_proba',
        n_jobs=-1
    )
    stacker_fold.fit(X_train_res, y_train_res)
    y5_cv_probas[val_idx] = stacker_fold.predict_proba(X_val_cv)[:, 1]
    print(f'Fold {fold+1}: val positives={y_val_cv.sum()}, avg proba bankrupt={y5_cv_probas[val_idx].mean():.4f}')

# Threshold tuning
print('\n--- Threshold Sweep ---')
best_thresh_5, best_acc_5 = 0.5, 0.0
for thresh in np.arange(0.02, 0.50, 0.01):
    preds = (y5_cv_probas >= thresh).astype(int)
    acc = custom_accuracy(y5, preds)
    _, rate = sparsity_check(preds)
    if rate < 0.20 and acc >= best_acc_5:
        best_acc_5 = acc
        best_thresh_5 = thresh
    if thresh in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
        print(f'  thresh={thresh:.2f}: acc={acc:.4f}, sparsity={rate:.4f}')

print(f'\nBest threshold: {best_thresh_5:.2f}')
y5_cv_preds = (y5_cv_probas >= best_thresh_5).astype(int)
evaluate_model(y5, y5_cv_preds, f'Cluster 5 — 5-Fold CV (thresh={best_thresh_5:.2f})')

Fold 1: val positives=10, avg proba bankrupt=0.2956
Fold 2: val positives=10, avg proba bankrupt=0.2853
Fold 3: val positives=10, avg proba bankrupt=0.3018
Fold 4: val positives=10, avg proba bankrupt=0.2728
Fold 5: val positives=10, avg proba bankrupt=0.3039

--- Threshold Sweep ---

Best threshold: 0.45
--- Cluster 5 — 5-Fold CV (thresh=0.45) ---
Custom Accuracy (TT/(TT+TF)): 0.6000
Sparsity: 0.1924 (PASS)
Confusion Matrix:
[[786 162]
 [ 20  30]]
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.83      0.90       948
           1       0.16      0.60      0.25        50

    accuracy                           0.82       998
   macro avg       0.57      0.71      0.57       998
weighted avg       0.93      0.82      0.86       998



(np.float64(0.6), np.float64(0.19238476953907815))

## Train Final Model & Save

In [26]:
# Train final model for cluster 5 with moderate SMOTE
smote_5 = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, int(y5.sum()) - 1),
                sampling_strategy=min(0.3, 1.0))
X5_res, y5_res = smote_5.fit_resample(X5_sel, y5)
print(f'After SMOTE: {len(X5_res)} samples (was {len(X5_sel)})')
print(f'Class distribution: {pd.Series(y5_res).value_counts().to_dict()}')

stacker_5 = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(n_estimators=80, max_depth=3, learning_rate=0.05, min_samples_leaf=10, random_state=RANDOM_STATE)),
        ('et', ExtraTreesClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=RANDOM_STATE),
    cv=3,
    stack_method='predict_proba',
    n_jobs=-1
)
stacker_5.fit(X5_res, y5_res)

# Training accuracy with tuned threshold
y5_train_proba = stacker_5.predict_proba(X5_sel)[:, 1]
y5_train_pred = (y5_train_proba >= best_thresh_5).astype(int)
evaluate_model(y5, y5_train_pred, f'Cluster 5 — Training (thresh={best_thresh_5:.2f})')

THRESHOLD_5 = best_thresh_5

After SMOTE: 1232 samples (was 998)
Class distribution: {0: 948, 1: 284}
--- Cluster 5 — Training (thresh=0.45) ---
Custom Accuracy (TT/(TT+TF)): 0.8800
Sparsity: 0.2265 (FAIL)
Confusion Matrix:
[[766 182]
 [  6  44]]
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.81      0.89       948
           1       0.19      0.88      0.32        50

    accuracy                           0.81       998
   macro avg       0.59      0.84      0.60       998
weighted avg       0.95      0.81      0.86       998



In [27]:
# Save cluster 5 model with threshold
model_5_info = {
    'model': stacker_5,
    'features': top_features_5,
    'n_features': N_FEATURES_5,
    'threshold': THRESHOLD_5,
    'cluster_id': 5,
    'model_type': 'stacking',
    'member': 'Noah Miller'
}
joblib.dump(model_5_info, os.path.join(SUBGROUP_MODEL_DIR, 'cluster_5_model.joblib'))
print(f'Cluster 5 model saved. Features: {N_FEATURES_5}, Threshold: {THRESHOLD_5:.2f}')

Cluster 5 model saved. Features: 10, Threshold: 0.45


## Summary Cluster 5

| Metric | CV (5-fold) | Training |
|---|---|---|
| Custom Accuracy (TT/(TT+TF)) | 60.00% (30/50) | 88.00% (44/50) |
| Sparsity (% predicted bankrupt) | 19.24% PASS | 22.65% FAIL |
| Features (N) | 10 | 10 |
| Threshold | 0.45 | 0.45 |
| Feature Score ((50-N)/50) | 0.80 | 0.80 |

**Estimated Rank Score:** 0.3(0.880) + 0.4(0.600) + 0.3(0.80) = 0.744

**Top 10 Features:** Borrowing dependency, Current Liabilities/Equity, Current Liability to Equity, Cash flow rate, Current Liability to Assets, Non-industry income and expenditure/revenue, Working capitcal Turnover Rate, Persistent EPS in the Last Four Seasons, Operating Funds to Liability, Net worth/Assets

**Model:** Stacking (RF + GB + ET → LR meta), moderate SMOTE (30%), 5-fold CV

**Note:** Training sparsity exceeds 20% but CV sparsity passes. The CV is the better estimate of test-set behavior. Sweep optimization improved rank from 0.730 (N=5) to 0.744 (N=10).